# Tour Interativo pelas 6 Etapas do `experimental-ml`

Este notebook guia novos pesquisadores e alunos de Iniciação Científica (IC) por todo o ciclo metodológico oficial em **6 etapas**, executando um experimento completo em classificação binária sem *data leakage*.

In [ ]:
# Instalação automática apenas quando executado no Google Colab
import subprocess, sys
if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/aleerangel/experimental-ml-framework.git",
    ])


## Etapa 1 — Ingestão Imutável, Diagnóstico e Saneamento

Carregamos dados tabulares com `load_dataset`, auditamos via `profile_dataset` e materializamos uma visão saneada imutável.

In [ ]:
from pathlib import Path
import tempfile
import numpy as np

from experimental_ml.core.enums import TaskType
from experimental_ml.data import (
    DatasetSchemaSpec,
    create_preparation_plan,
    load_dataset,
    profile_dataset,
    sanitize_dataset,
)

rng = np.random.default_rng(42)
dados = {
    'feat_a': rng.normal(0.0, 1.0, size=36).tolist(),
    'feat_b': rng.normal(1.5, 0.7, size=36).tolist(),
    'alvo': [0 if i < 18 else 1 for i in range(36)],
}
schema = DatasetSchemaSpec(target_col='alvo', task_type=TaskType.BINARY_CLASSIFICATION)
bundle = load_dataset(dados, schema)
profile = profile_dataset(bundle)
plan = create_preparation_plan(profile, rules=[])
view = sanitize_dataset(bundle, profile, plan)
print(f'Amostras: {len(view.observation_ids)}, Hash: {view.dataset_hash[:12]}...')

## Etapa 2 e Etapa 3 — Cenários de Features e Catálogo de Modelos

Declaramos um cenário fixo (`ScenarioMode.FIXED`) e validamos dois modelos baseline.

In [ ]:
from experimental_ml.features import FeatureScenario, ScenarioMode, validate_scenarios
from experimental_ml.models import (
    DummyBaselineModel,
    ModelCapabilities,
    ModelFamily,
    ModelSpec,
    validate_model_spec,
)

cenarios = validate_scenarios([
    FeatureScenario(
        scenario_id='todas',
        name='Todas as Features',
        description='Conjunto completo de preditores',
        scientific_justification='Baseline sem seleção',
        mode=ScenarioMode.FIXED,
        selected_features=('feat_a', 'feat_b'),
    )
], view)

modelos = [
    validate_model_spec(ModelSpec(
        model_id='dummy_freq',
        display_name='Dummy Most Frequent',
        family=ModelFamily.BASELINE,
        estimator_factory=lambda seed=None: DummyBaselineModel(strategy='most_frequent'),
        capabilities=ModelCapabilities(supports_predict_proba=True),
        description='Classificador majoritário',
        scientific_justification='Referência mínima',
    ))
]
print('Cenários e modelos validados com sucesso.')

## Etapa 4 — Particionamento Compartilhado (4A) e Execução Matricial (4B)

Construímos o `PartitionPlan` determinístico e executamos a matriz completa com persistência atômica.

In [ ]:
from experimental_ml.orchestration import load_cell_artifacts, run_experiment_matrix
from experimental_ml.validation import (
    EvaluationRegime,
    PartitionConfig,
    PartitionStrategy,
    build_partition_plan,
)

tmp_dir = Path(tempfile.mkdtemp(prefix='expml_nb01_'))
part_plan = build_partition_plan(
    view,
    PartitionConfig(
        strategy=PartitionStrategy.STRATIFIED_KFOLD,
        regime=EvaluationRegime.CV_ONLY,
        n_splits=4,
        master_seed=42,
    ),
)
run_state = run_experiment_matrix(
    model_specs=modelos,
    scenarios=cenarios.scenarios,
    partition_plan=part_plan,
    sanitized_view=view,
    run_dir=tmp_dir / 'runs',
    metrics=('accuracy', 'f1'),
    master_seed=42,
    overwrite=True,
)
print(f'Células concluídas: {run_state.completed_cells}/{run_state.total_cells}')

## Etapa 5 e Etapa 6 — Consolidação Estatística e Relatório Offline

Consolidamos as métricas sem coerção silenciosa para zero e geramos uma tabela em Markdown.

In [ ]:
from experimental_ml.statistics import consolidate_descriptive

cell_results = [load_cell_artifacts(cid, tmp_dir / 'runs')[0] for cid in run_state.cell_statuses]
consolidation = consolidate_descriptive(
    cell_results,
    total_folds_per_config=len(part_plan.folds),
)
cfg_id, summary = next(iter(consolidation.configurations.items()))
acc = summary.aggregated_metrics['accuracy']
print(f'Configuração {cfg_id}: cobertura={summary.coverage_status.value}, acurácia média={acc.mean:.4f}')
assert summary.coverage_status.value == 'VALID'